In [2]:
import json
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

In [3]:
# Paths
IMAGE_DIR = "../data/train_images"
ANNOTATION_PATH = "../data/train_annotations.json"

print("Image dir exists:", os.path.exists(IMAGE_DIR))
print("Annotation file exists:", os.path.exists(ANNOTATION_PATH))

# Load annotations
with open(ANNOTATION_PATH, "r") as f:
    train_annotations = json.load(f)

print("Loaded annotations:", len(train_annotations))


Image dir exists: True
Annotation file exists: True
Loaded annotations: 9580


In [4]:
def sanitize_boxes(boxes, labels):
    """
    Remove invalid bounding boxes (zero or negative width/height)
    boxes: Tensor [N, 4] in (xmin, ymin, xmax, ymax)
    labels: Tensor [N]
    """
    x_min, y_min, x_max, y_max = boxes.unbind(1)

    keep = (x_max > x_min) & (y_max > y_min)

    boxes = boxes[keep]
    labels = labels[keep]

    return boxes, labels


In [5]:
class DamageDataset(Dataset):
    def __init__(self, images_dir, annotations, transforms=None):
        self.images_dir = images_dir
        self.annotations = annotations
        self.transforms = transforms

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_path = os.path.join(self.images_dir, ann["image_name"])

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            return None

        boxes = torch.tensor(ann["boxes"], dtype=torch.float32)
        labels = torch.tensor(ann["labels"], dtype=torch.int64)

        boxes, labels = sanitize_boxes(boxes, labels)

        if boxes.shape[0] == 0:
            return None

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64)
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target


In [6]:
def collate_fn(batch):
    # Remove samples that are None
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch))


In [7]:
def get_transform():
    return T.Compose([
        T.ToTensor()
    ])


In [9]:
# Create dataset
dataset = DamageDataset(
    images_dir=IMAGE_DIR,
    annotations=train_annotations,
    transforms=get_transform()
)

print("Dataset size:", len(dataset))

# Create dataloader
loader = DataLoader(
    dataset,
    batch_size=4,          # keep small for detection models
    shuffle=True,
    collate_fn=collate_fn
)

# Sanity check: load one batch
images, targets = next(iter(loader))

print("Batch loaded successfully")
print("Number of images in batch:", len(images))
print("Boxes in first image:", targets[0]["boxes"].shape)


Dataset size: 9580
Batch loaded successfully
Number of images in batch: 4
Boxes in first image: torch.Size([2, 4])
